In [ ]:
import os
import types
import pandas as pd
from botocore.client import Config
import ibm_boto3
from project_lib import Project

# =============================================================================
# CONFIGURATION DE LA CONNEXION À IBM CLOUD OBJECT STORAGE
# =============================================================================

def __iter__(self): 
    """Fonction auxiliaire pour rendre les objets itérables"""
    return 0

# Création du client IBM Cloud Object Storage
cos_client = ibm_boto3.client(
    service_name='s3',
    ibm_api_key_id='osPVpdg1TF0CVgPLwQV9XqraQm3mZ_Pc9i4p35_ZmZoq',
    ibm_auth_endpoint="https://iam.cloud.ibm.com/identity/token",
    config=Config(signature_version='oauth'),
    endpoint_url='https://s3.direct.ca-tor.cloud-object-storage.appdomain.cloud'
)

# =============================================================================
# DÉFINITION DES FICHIERS À CHARGER
# =============================================================================

# Nom du bucket contenant les données
BUCKET_NAME = 'hackatonbipipelinesubstainability-donotdelete-pr-ieeieg8rpoe1xe'

# Liste complète des fichiers CSV à charger depuis le bucket
# Ces fichiers contiennent des données d'évaluation de modèles LLM
# avec différentes configurations (modèles, datasets, machines)
csv_files_to_load = [
    "alpaca_gemma_2b_workstation.csv",
    "alpaca_gemma_2b_laptop2.csv",
    "alpaca_gemma_7b_workstation.csv",
    "alpaca_gemma_7b_laptop2.csv",
    "alpaca_gemma_2b_laptop1.csv",
    "codefeedback_gemma_7b_laptop2.csv",
    "codefeedback_gemma_2b_workstation.csv",
    "codefeedback_codellama_7b_laptop1.csv",
    "codefeedback_codellama_7b_laptop2.csv",
    "codefeedback_gemma_7b_workstation.csv",
    "codefeedback_codellama_7b_workstation.csv",
    "codefeedback_gemma_2b_laptop2.csv",
    "codefeedback_codellama_70b_workstation.csv",
    "alpaca_llama3_8b_laptop2.csv",
    "alpaca_llama3_70b_server.csv"
]

# =============================================================================
# FONCTION D'EXTRACTION DU TYPE DE MACHINE
# =============================================================================

def extract_LLM_type(filename):
    """
    Extrait le nom du modèle, le nombre de paramètre et le type de machine depuis le nom du fichier CSV.
    
    Exemple:
        "alpaca_gemma_2b_workstation.csv" → ('gemma', '2b', 'workstation')
    """
    filename_parts = filename.split('_')
    return (filename_parts[1],filename_parts[2],filename_parts[3])

# =============================================================================
# CHARGEMENT DES FICHIERS CSV
# =============================================================================

print("=" * 80)
print("ÉTAPE 1 : CHARGEMENT DES FICHIERS CSV")
print("=" * 80)

# Liste qui contiendra tous les DataFrames chargés
list_of_dataframes = []

# Boucle de chargement : on récupère chaque fichier CSV du bucket
for csv_filename in csv_files_to_load:
    # Récupération du fichier depuis le bucket IBM Cloud Object Storage
    file_object = cos_client.get_object(Bucket=BUCKET_NAME, Key=csv_filename)
    file_body = file_object['Body']
    
    # Correction technique : rendre l'objet itérable pour pandas
    if not hasattr(file_body, "__iter__"):
        file_body.__iter__ = types.MethodType(__iter__, file_body)
    
    # Lecture du CSV dans un DataFrame pandas
    dataframe = pd.read_csv(file_body)

    # Ajout de la colonne 'machine_type' basée sur le nom du fichier
    LLM_type = extract_LLM_type(csv_filename)
    dataframe['model_name_split'], dataframe['model_size'], dataframe['machine_type'] = LLM_type
    
    list_of_dataframes.append(dataframe)
    
    # Affichage des informations de chargement
    print(f"✅ {csv_filename:<50} | {dataframe.shape[0]:>6} lignes × {dataframe.shape[1]:>3} colonnes")

print(f"\n📊 Total de fichiers chargés : {len(list_of_dataframes)}")

# =============================================================================
# FUSION DES DATAFRAMES
# =============================================================================

print("\n" + "=" * 80)
print("ÉTAPE 2 : FUSION DES DATAFRAMES")
print("=" * 80)

# Identification des colonnes communes à tous les DataFrames
# On commence avec les colonnes du premier DataFrame
common_columns = set(list_of_dataframes[0].columns)

# On fait l'intersection avec les colonnes de chaque DataFrame suivant
for df in list_of_dataframes[1:]:
    common_columns &= set(df.columns)

print(f"\n📋 Colonnes communes identifiées : {len(common_columns)} colonnes")
print(f"   {sorted(common_columns)}")

# Filtrage : on ne garde que les colonnes communes dans chaque DataFrame
# Cela assure que tous les DataFrames ont la même structure avant la fusion
list_of_dataframes = [df[list(common_columns)] for df in list_of_dataframes]

# Concaténation verticale de tous les DataFrames en un seul
combined_dataset = pd.concat(list_of_dataframes, ignore_index=True)

print(combined_dataset["machine_type"].value_counts())

print(f"\n✅ Dataset combiné créé : {combined_dataset.shape[0]} lignes × {combined_dataset.shape[1]} colonnes")



In [ ]:
# =============================================================================
# SPLIT DU NOM DU MODEL
# =============================================================================
combined_dataset.drop('model_name', axis=1, inplace=True)
combined_dataset['model_size']=(combined_dataset['model_size'].str.replace('b', '', regex=False).astype(float)*1000000000)
print(combined_dataset['model_size'].head(10))

In [ ]:
# =============================================================================
# NETTOYAGE DES DONNÉES
# =============================================================================

print("\n" + "=" * 80)
print("ÉTAPE 3 : NETTOYAGE DES DONNÉES")
print("=" * 80)

# Analyse des valeurs manquantes
print("\n🔍 Analyse des valeurs manquantes :")
missing_values_count = combined_dataset.isna().sum()
columns_with_missing = missing_values_count[missing_values_count > 0]

if len(columns_with_missing) > 0:
    print(f"   {len(columns_with_missing)} colonnes contiennent des valeurs manquantes")
    for col, count in columns_with_missing.items():
        print(f"   - {col}: {count} valeurs manquantes")
else:
    print("   Aucune valeur manquante détectée")

# Suppression des lignes contenant des valeurs manquantes
rows_before = combined_dataset.shape[0]
combined_dataset = combined_dataset.dropna()
rows_after = combined_dataset.shape[0]
rows_removed = rows_before - rows_after

print(f"\n🧹 Suppression des lignes avec valeurs manquantes : {rows_removed} lignes supprimées")

# =============================================================================
# SÉLECTION DES COLONNES PERTINENTES
# =============================================================================

print("\n" + "=" * 80)
print("ÉTAPE 4 : SÉLECTION DES COLONNES PERTINENTES")
print("=" * 80)

# Colonnes à supprimer car non pertinentes pour l'analyse
# - Colonnes d'index redondantes
# - Colonnes temporelles déjà agrégées ailleurs
# - Colonnes de texte brut trop volumineuses
# - Métriques énergétiques intermédiaires (on garde seulement le total)
# - Métriques linguistiques peu corrélées
columns_to_drop = [
    'Unnamed: 0',                      # Index automatique pandas
    'index',                           # Index redondant
    'created_at',                      # Timestamp de création
    'prompt',                          # Texte du prompt (trop volumineux)
    'response',                        # Texte de la réponse (trop volumineux)
    'type',                            # Type de donnée
    'clock_duration',                  # Durée (redondant avec d'autres métriques)
    'start_time',                      # Heure de début
    'end_time',                        # Heure de fin
    'energy_consumption_llm',          # Consommation partielle (on garde le total)
    'energy_consumption_monitoring',   # Consommation du monitoring
    'energy_consumption_llm_cpu',      # Consommation CPU seule
    'energy_consumption_llm_gpu',      # Consommation GPU seule
    'adjectives',                      # Nombre d'adjectifs (faible corrélation)
    'adverbs'                          # Nombre d'adverbes (faible corrélation)
]

# Suppression des colonnes (si elles existent dans le dataset)
existing_columns_to_drop = [col for col in columns_to_drop if col in combined_dataset.columns]
combined_dataset.drop(columns=existing_columns_to_drop, inplace=True)

print(f"🗑️  {len(existing_columns_to_drop)} colonnes supprimées")
print(f"📊 Dataset après nettoyage : {combined_dataset.shape[0]} lignes × {combined_dataset.shape[1]} colonnes")

# =============================================================================
# ENCODAGE DES VARIABLES CATÉGORIELLES
# =============================================================================

print("\n" + "=" * 80)
print("ÉTAPE 5 : ENCODAGE DES VARIABLES CATÉGORIELLES")
print("=" * 80)

# Encodage de la colonne 'text_standard' (niveau de lisibilité du texte)
# Transformation en codes numériques pour permettre l'utilisation dans des modèles ML
if 'text_standard' in combined_dataset.columns:
    combined_dataset['text_standard'] = combined_dataset['text_standard'].astype('category').cat.codes
    print("✅ Colonne 'text_standard' encodée en valeurs numériques")

# Encodage de la colonne 'model_name' (nom du modèle LLM utilisé)
# Transformation en codes numériques pour l'analyse
if 'model_name_split' in combined_dataset.columns:
    combined_dataset['model_name_split'] = combined_dataset['model_name_split'].astype('category').cat.codes
    print("✅ Colonne 'model_name_split' encodée en valeurs numériques")

# Encodage de la colonne 'machine_type' (nom de la machine utilisée)
# Transformation en codes numériques pour l'analyse
if 'machine_type' in combined_dataset.columns:
    combined_dataset['machine_type'] = combined_dataset['machine_type'].astype('category').cat.codes
    print("✅ Colonne 'machine_type' encodée en valeurs numériques")

In [ ]:
# =============================================================================
# SÉLECTION DES FEATURES PAR CORRÉLATION
# =============================================================================

print("\n" + "=" * 80)
print("ÉTAPE 6 : SÉLECTION DES FEATURES PAR CORRÉLATION")
print("=" * 80)

# Variable cible : consommation énergétique totale du LLM
TARGET_COLUMN = 'energy_consumption_llm_total'

# Calcul de la corrélation de chaque colonne avec la variable cible
if TARGET_COLUMN in combined_dataset.columns:
    correlation_with_target = combined_dataset.corrwith(
        combined_dataset[TARGET_COLUMN]
    ).sort_values(ascending=False)
    
    print(f"\n📈 Corrélations avec '{TARGET_COLUMN}' :")
    for col, corr in correlation_with_target.items():
        print(f"   {col:<40} : {corr:>7.4f}")
    
    # Seuil de corrélation : on garde seulement les colonnes avec |corrélation| >= 0.07
    # Cela permet de réduire la dimensionnalité en éliminant les features peu informatives
    CORRELATION_THRESHOLD = 0.09
    
    columns_to_keep = correlation_with_target[
        correlation_with_target.abs() >= CORRELATION_THRESHOLD
    ].index.tolist()
    
    # Filtrage du dataset
    combined_dataset = combined_dataset[columns_to_keep]
    
    print(f"\n✂️  Seuil de corrélation appliqué : |r| >= {CORRELATION_THRESHOLD}")
    print(f"📊 Features conservées : {len(columns_to_keep)} colonnes")
    print(f"   {columns_to_keep}")
else:
    print(f"⚠️  Attention : La colonne cible '{TARGET_COLUMN}' est introuvable")

# =============================================================================
# EXPORT DU DATASET FINAL
# =============================================================================

print("\n" + "=" * 80)
print("ÉTAPE 7 : EXPORT DU DATASET FINAL")
print("=" * 80)

# Configuration du projet Watson
watson_project = Project(
    project_id='5d16b795-9991-462e-b9c6-f0f06b8e5811',
    project_access_token='p-2+cl/ScTAKcB8qhT1HXIaEEA==;iVdjXn90yP+w8QFxYy8fdw==:p7wvdoU+tnyVEANHaqRmAKg0U6rPfh/qeLXQ2QPJd8P1L3DfG1XTAALkTal8uTXJvSh0Ucw/ua4PfaM/iLOxT4+02mcsHZDBmg=='
)

# Export du dataset nettoyé et prétraité
OUTPUT_FILENAME = 'data.csv'

watson_project.save_data(
    data=combined_dataset.to_csv(index=False),
    file_name=OUTPUT_FILENAME,
    overwrite=True
)

print(f"✅ Dataset exporté avec succès dans le projet Watson")
print(f"📁 Nom du fichier : {OUTPUT_FILENAME}")
print(f"📊 Dimensions finales : {combined_dataset.shape[0]} lignes × {combined_dataset.shape[1]} colonnes")

# Affichage d'un aperçu du dataset final
print("\n" + "=" * 80)
print("APERÇU DU DATASET FINAL")
print("=" * 80)
print(combined_dataset.head())

print("\n" + "=" * 80)
print("✨ TRAITEMENT TERMINÉ AVEC SUCCÈS")
print("=" * 80)